# Hawkes $k$-Point Function — Time-Domain Integration

Streamlined notebook: model → propagator → diagrams → Phase J time-domain integration → simulation comparison.
No frequency-domain Phase I code.


In [ ]:
%display latex
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

from IPython.display import display, Math
from sage.all import (
    SR, matrix, latex, I, pi, exp, diff, solve, integrate, oo,
    dirac_delta, function, var
)

from msrjd.core.field_theory import FieldTheory, fourier_transform, inverse_fourier_transform
from msrjd.core.vertices import extract_vertex_types, extract_source_types, available_degrees
from msrjd.core.cache import PipelineCache
from msrjd.enumeration.loop_diagram_enumeration import enumerate_all
from msrjd.diagrams.filter import filter_prediagrams, classify_prediagram_vertices
from msrjd.diagrams.type_assignment import (
    enumerate_typed_diagrams, enumerate_all as enumerate_all_typed,
    build_field_index_map, TypedDiagram
)
from msrjd.diagrams.causality import filter_causal
from msrjd.diagrams.symmetry import (
    combinatorial_factor, compute_all_combinatorial_factors,
    deduplicate_typed_diagrams,
)

from models.hawkes_linear_sage import HAWKES_MODEL

print('Imports loaded.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Configuration
# ══════════════════════════════════════════════════════════════
k       = 3      # k-point cumulant (number of external legs)
max_ell = 1      # maximum loop order (0 = tree only, 1 = tree + 1-loop, ...)

# External fields: one per external leg (must have length k).
# Pick from the physical fields of the model: ('dn', pop), ('dv', pop)
# where pop is a population index (1, 2, ...).
# Examples:
#   k=1:  [('dn', 1)]
#   k=2:  [('dn', 1), ('dn', 2)]
#   k=2:  [('dn', 1), ('dv', 1)]    (mixed correlator)
external_fields = [('dn', 1), ('dn', 1), ('dn', 2)]

USE_CACHE = True  # True: pull from cache when available; False: recompute all

assert len(external_fields) == k, f'external_fields has {len(external_fields)} entries but k={k}'
print(f'k = {k},  max loop order = {max_ell}')
print(f'External fields: {external_fields}')

---
## 1. Theory / Model Information

In [ ]:
m = HAWKES_MODEL
print(f"Model:  {m['name']}")
print(f"Convention: {m['background_rate_convention']}")
print()

print('── Index sets ──')
for name, vals in m['index_sets'].items():
    print(f'  {name}: {list(vals)}')
print()

print('── Response fields (not expanded — full integration variables) ──')
for f in m['response_fields']:
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Physical fields (expanded around MF background) ──')
for f in m['physical_fields']:
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Parameters ──')
for p in m.get('parameters', []):
    idx = '(indexed)' if p.get('indexed') else '(scalar)'
    dom = f", domain={p['domain']}" if p.get('domain') else ''
    print(f"  {p['name']}  {idx}{dom}  — {p.get('description', '')}")
print()

print('── Kernels ──')
for kern in m.get('kernels', []):
    print(f"  {kern['name']}  — {kern.get('description', '')}")
print()

print('── Operators ──')
for o in m.get('operators', []):
    print(f"  {o['name']}  (latex: ${o.get('latex_name', o['name'])}$)  — {o.get('description', '')}")
print()

print('── Nonlinear functions ──')
for f in m.get('functions', []):
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Specializations ──')
print('  φ quadratic (cubic and higher coefficients = 0)')
print('  g = δ(t)  (instantaneous synaptic coupling)')

### 1.1 Expand the action and show all sectors

In [3]:
ft = FieldTheory(HAWKES_MODEL, taylor_order=4)
ft.expand()

R  = ft.ring()
ns = ft._ns

print(f'Taylor order: {ft.taylor_order}')
print(f'Polynomial ring: {R}')
print(f'Ring generators: {R.gens()}')
print(f'Response generators (n_tilde={ft._n_tilde}): {R.gens()[:ft._n_tilde]}')
print(f'Physical generators: {R.gens()[ft._n_tilde:]}')
print()

passed = ft.sanity_check()
print()
ft.summary()

Taylor order: 4
Polynomial ring: Multivariate Polynomial Ring in nt1, nt2, vt1, vt2, dn1, dn2, dv1, dv2 over Symbolic Ring
Ring generators: (nt1, nt2, vt1, vt2, dn1, dn2, dv1, dv2)
Response generators (n_tilde=4): (nt1, nt2, vt1, vt2)
Physical generators: (dn1, dn2, dv1, dv2)

=== Sanity checks ===
  [PASS]  (n_tilde=0, n_phys=0)  constant term
  [PASS]  (n_tilde=1, n_phys=0)  tadpole — must vanish at MF saddle
  [PASS]  (n_tilde=0, n_phys=1)  linear physical-only — must vanish at EOM

=== Action sectors ===
  (n_tilde=1, n_phys=1)  [free action]:


<IPython.core.display.Math object>

  (n_tilde=1, n_phys=2)  [vertex (order 3)]:


<IPython.core.display.Math object>

  (n_tilde=2, n_phys=0)  [noise kernel]:


...

<IPython.core.display.Math object>

  (n_tilde=4, n_phys=1)  [vertex (order 5)]:


<IPython.core.display.Math object>

  (n_tilde=4, n_phys=2)  [vertex (order 6)]:


<IPython.core.display.Math object>

### 1.2 Propagator

Extract the kernel matrix $\mathbf{K}$ from the free action, Fourier transform, and invert.

In [ ]:
import signal

S_free = ft.free_action()
ring_gen_names = [str(g) for g in R.gens()]

# Use ring variable ordering (must match build_field_index_map)
resp_names = ring_gen_names[:ft._n_tilde]
phys_names = ring_gen_names[ft._n_tilde:]

pos_to_row = {ring_gen_names.index(nm): i for i, nm in enumerate(resp_names)}
pos_to_col = {ring_gen_names.index(nm): j for j, nm in enumerate(phys_names)}

nf = len(resp_names)
K_data = [[SR(0)] * nf for _ in range(nf)]
for exp_vec, coeff in S_free.dict().items():
    row = col = None
    for idx in range(len(ring_gen_names)):
        if exp_vec[idx] > 0:
            if idx in pos_to_row: row = pos_to_row[idx]
            if idx in pos_to_col: col = pos_to_col[idx]
    if row is not None and col is not None:
        K_data[row][col] += SR(coeff)

K_mat = matrix(SR, K_data)

# Convert to kernel form
Dt       = ns.Dt
delta_D  = ns.delta_D
delta_Dp = ns.delta_Dp

def _to_kernel(c):
    c = SR(c)
    if c.has(delta_D) or c.has(ns.g):
        return c
    p0   = c.subs({Dt: 0})
    rest = (c - p0).subs({Dt: delta_Dp})
    return p0 * delta_D + rest

K_ker = matrix(SR, [[_to_kernel(K_mat[i, j]) for j in range(nf)] for i in range(nf)])

resp_sr  = [ns.nt[i] for i in ns.pop] + [ns.vt[i] for i in ns.pop]
phys_sr  = [ns.dn[i] for i in ns.pop] + [ns.dv[i] for i in ns.pop]

print('Field ordering:')
display(Math(r'\tilde{\mathbf{a}} = ' + latex(vector(resp_sr))
             + r',\qquad \mathbf{a} = ' + latex(vector(phys_sr))))
print()
print('Kernel matrix K (time domain):')
display(Math(r'\mathbf{K} = ' + latex(K_ker)))

# Fourier transform
t_var = SR.var('t')
omega = SR.var('omega', latex_name=r'\omega')

time_domain = {
    delta_D:  dirac_delta(t_var),
    delta_Dp: diff(dirac_delta(t_var), t_var),
    ns.g:     function('g')(t_var),
}

K_ft_data = [[SR(0)] * nf for _ in range(nf)]
for i in range(nf):
    for j in range(nf):
        c = K_ker[i, j]
        if not c.is_zero():
            K_ft_data[i][j] = fourier_transform(SR(c).subs(time_domain), t_var, omega)

K_ft = matrix(SR, K_ft_data)
print('Fourier-domain kernel:')
display(Math(r'\hat{\mathbf{K}}(\omega) = ' + latex(K_ft)))

# Propagator inverse
G_ft = K_ft.inverse().apply_map(lambda e: e.factor())
G_ft_explicit = True
print('Propagator:')
display(Math(r'\hat{\mathbf{G}}(\omega) = ' + latex(G_ft)))

# Adjugate, determinant, poles
adj_ft  = K_ft.adjugate()
D_omega = K_ft.det().expand()
D_prime = diff(D_omega, omega)

pole_eqs = solve(D_omega == 0, omega)
pole_vals = [eq.rhs() if hasattr(eq, 'rhs') else eq for eq in pole_eqs]

print(f'\ndet(K̂) = {D_omega}')
print(f'Poles ({len(pole_vals)}):')
for pk, p in enumerate(pole_vals):
    display(Math(r'\omega_{' + str(pk+1) + '} = ' + latex(p)))

# Residue matrices
C_mats = []
for omega_k in pole_vals:
    C_data = [[SR(0)] * nf for _ in range(nf)]
    for i in range(nf):
        for j in range(nf):
            n_ij = adj_ft[i, j]
            if not n_ij.is_zero():
                num = n_ij.subs({omega: omega_k})
                den = D_prime.subs({omega: omega_k})
                C_data[i][j] = (I * num / den).factor()
    C_mats.append(matrix(SR, C_data))

print('Residue matrices:')
for pk, C in enumerate(C_mats):
    display(Math(r'\mathbf{C}_{' + str(pk+1) + r'} = ' + latex(C)))

# Time-domain propagator (smooth part)
G_t = sum(C_mats[pk] * exp(I * pole_vals[pk] * t_var) for pk in range(len(pole_vals)))
G_t = G_t.apply_map(lambda e: e.simplify_full())

# Delta-function coefficients via polynomial division:
#   G_ft[i,j](ω) = Q(iω) + R(ω)/D(ω)
# where Q is the polynomial (non-proper) part and R/D is strictly proper.
# Q maps to distributional terms: constant → δ(t), iω → δ'(t), etc.
# The simplest case: D[i,j] = lim_{ω→∞} G_ft[i,j](ω) = coefficient of δ(t).
# Use Sage's symbolic limit — works with symbolic parameters, no numerics needed.
from sage.all import limit as _limit, oo as _oo
D_delta_data = [[SR(0)] * nf for _ in range(nf)]
for i in range(nf):
    for j in range(nf):
        entry = SR(G_ft[i, j])
        if entry.is_zero():
            continue
        try:
            lim_val = _limit(entry, **{str(omega): _oo})
            if not lim_val.is_zero():
                D_delta_data[i][j] = lim_val
        except Exception:
            pass
D_delta = matrix(SR, D_delta_data)

print('Delta-function coefficient matrix:')
display(Math(r'\mathbf{D} = ' + latex(D_delta)))

print()
print('Full retarded propagator:')
display(Math(
    r'\mathbf{G}^R(t) = \mathbf{D}\,\delta(t)'
    r' \;+\; \Theta(t) \sum_{k=1}^{' + str(len(pole_vals)) + r'}'
    r' \mathbf{C}_k \, e^{i\omega_k t}'
))

---
## 2. Prediagram Enumeration

Enumerate all trees, topologies, and prediagrams for the $k$-point function
at each loop order $\ell = 0, \ldots, \ell_{\max}$.

In [5]:
from sage.plot.plot import graphics_array

def plot_prediagrams(pds, title_prefix=''):
    """Plot prediagrams with colored vertices."""
    if not pds:
        print('  (none)')
        return
    plots = []
    for i, (D, G, leaves, internal) in enumerate(pds):
        leaves_set = set(leaves)
        color_map = {}
        for v in D.vertices():
            if v in leaves_set:
                color_map.setdefault('black', []).append(v)
            elif D.in_degree(v) == 0:
                color_map.setdefault('red', []).append(v)
            else:
                color_map.setdefault('lightblue', []).append(v)
        plots.append(D.plot(vertex_colors=color_map, vertex_size=400,
                            edge_thickness=2, title=f"{title_prefix}{i+1}"))
    n_cols = min(4, len(plots))
    n_rows = (len(plots) + n_cols - 1) // n_cols
    graphics_array(plots, n_rows, n_cols).show(figsize=[5*n_cols, 4*n_rows])

def plot_topologies(topos, title_prefix=''):
    """Plot undirected topologies."""
    if not topos:
        print('  (none)')
        return
    plots = []
    for i, (G, leaves, internal) in enumerate(topos):
        leaves_set = set(leaves)
        color_map = {}
        for v in G.vertices():
            if v in leaves_set:
                color_map.setdefault('black', []).append(v)
            else:
                color_map.setdefault('lightgray', []).append(v)
        plots.append(G.plot(vertex_colors=color_map, vertex_size=400,
                            edge_thickness=2, title=f"{title_prefix}{i+1}"))
    n_cols = min(4, len(plots))
    n_rows = (len(plots) + n_cols - 1) // n_cols
    graphics_array(plots, n_rows, n_cols).show(figsize=[5*n_cols, 4*n_rows])

print('Plot helpers defined.')

Plot helpers defined.


In [ ]:
# ── Pipeline cache (depends on model + k) ──────────────────────
# Model name + external fields sanitized for filesystem use
_model_tag = HAWKES_MODEL['name'].replace(' ', '_').lower()
_ext_tag = '_'.join(f'{f[0]}{f[1]}' for f in external_fields)
cache_dir = f'saved_results/{_model_tag}_k{k}_{_ext_tag}'
cache = PipelineCache(cache_dir)
if USE_CACHE:
    print(f'Cache: {cache}')
    for e in cache.list_cached():
        print(f'  {e["key"]}  (saved {e["saved_at"][:19]})')
else:
    print('Cache disabled — all stages will be recomputed.')

# ── Enumerate prediagrams for each loop order ──────────────────
pds_by_ell = {}    # ell -> list of prediagrams
counts_by_ell = {} # ell -> counts dict

for ell in range(max_ell + 1):
    def _enumerate(ell=ell):
        t, tp, pd, c = enumerate_all(k, ell, verbose=False)
        return (pd, c)

    pds, counts = cache.get_or_compute(
        'prediagrams', _enumerate, k=k, loop_order=ell
    ) if USE_CACHE else _enumerate()

    pds_by_ell[ell] = pds
    counts_by_ell[ell] = counts

    print(f'ell={ell}:  {counts["n_trees"]} trees,  '
          f'{counts["n_topologies"]} topologies,  '
          f'{counts["n_prediagrams"]} prediagrams')
    plot_prediagrams(pds, title_prefix=f'ell={ell} PD ')

---
## 3. Vertex Extraction

Extract typed vertices (`VertexType`, `SourceType`) from the expanded action, based on the maximum vertex degree required by the prediagrams.

In [ ]:
from msrjd.enumeration.degree_scan import max_vertex_degree, scan_source_vertices

all_pds = [pd for pds in pds_by_ell.values() for pd in pds]
max_deg = max_vertex_degree(all_pds)
src_degs = scan_source_vertices(all_pds)
print(f'Max vertex degree across all prediagrams: {max_deg}')
print(f'Source vertex out-degrees needed: {src_degs}')
print(f'Current Taylor order: {ft.taylor_order}')
print(f'Sufficient: {ft.taylor_order >= max_deg}')
print()

vtypes = extract_vertex_types(ft)
stypes = extract_source_types(ft)

print(f'── Interaction vertex types ({len(vtypes)}) ──')
for i, vt in enumerate(vtypes):
    print(f'  V{i+1}: bigrade={vt.bigrade}, in_deg={vt.in_degree}, out_deg={vt.out_degree}')
    print(f'        resp_legs={vt.response_legs}, phys_legs={vt.physical_legs}')
    display(Math(f'\\quad\\text{{coeff}} = {latex(vt.coefficient)}'))

print(f'\n── Source types ({len(stypes)}) ──')
for i, st in enumerate(stypes):
    print(f'  S{i+1}: bigrade={st.bigrade}, out_deg={st.out_degree}')
    print(f'        resp_legs={st.response_legs}')
    display(Math(f'\\quad\\text{{coeff}} = {latex(st.coefficient)}'))

int_degs, src_degs_avail = available_degrees(vtypes, stypes)
print(f'\nAvailable interaction degrees (in, out): {sorted(int_degs)}')
print(f'Available source out-degrees: {sorted(src_degs_avail)}')

---
## 4. Prediagram Filtering

Remove prediagrams whose vertex degree signatures don't match any available vertex or source type.

In [ ]:
kept_by_ell = {}  # ell -> list of kept prediagrams

for ell in range(max_ell + 1):
    def _filter(ell=ell):
        kept, disc = filter_prediagrams(pds_by_ell[ell], vtypes, stypes)
        return (kept, disc)

    kept, disc = cache.get_or_compute(
        'filtered', _filter, k=k, loop_order=ell
    ) if USE_CACHE else _filter()

    kept_by_ell[ell] = kept
    print(f'ell={ell}:  {len(pds_by_ell[ell])} prediagrams -> '
          f'{len(kept)} kept, {disc} discarded')
    plot_prediagrams(kept, title_prefix=f'ell={ell} PD ')

---
## 5. Unique Typed Diagrams

For each surviving prediagram, enumerate all valid field-type assignments,
filter for causality, and deduplicate by isomorphism to obtain the set of
**unique Feynman diagrams** $\Gamma$.  Only these are cached — intermediate
typed diagrams are not stored.

In [ ]:
# External fields are specified in the Configuration cell above.
print(f'External fields (k={k}): {external_fields}')

# Build field index maps
ring_var_names_list = list(ns._ring_var_names)
n_tilde = ft._n_tilde
resp_idx, phys_idx = build_field_index_map(ring_var_names_list, n_tilde)

# Validate that each external field exists in the physical field index
for field in external_fields:
    assert field in phys_idx, (
        f'External field {field} not found in physical fields. '
        f'Available: {sorted(phys_idx.keys())}'
    )

print('\nResponse field index map:')
for leg, idx in sorted(resp_idx.items(), key=lambda x: x[1]):
    print(f'  {leg} -> row {idx}')
print()
print('Physical field index map:')
for leg, idx in sorted(phys_idx.items(), key=lambda x: x[1]):
    print(f'  {leg} -> col {idx}')

In [ ]:
unique_by_ell = {}  # ell -> list of unique TypedDiagram

for ell in range(max_ell + 1):
    def _unique(ell=ell):
        typed = enumerate_all_typed(
            kept_by_ell[ell], external_fields, vtypes, stypes,
            G_ft, resp_idx, phys_idx
        )
        causal, n_disc, _ = filter_causal(typed)
        unique = deduplicate_typed_diagrams(causal)
        print(f'  ell={ell}: {len(kept_by_ell[ell])} prediagrams -> {len(typed)} typed'
              f' -> {len(causal)} causal -> {len(unique)} unique')
        return unique

    unique_by_ell[ell] = cache.get_or_compute(
        'unique_typed', _unique, k=k, loop_order=ell
    ) if USE_CACHE else _unique()

    print(f'ell={ell}: {len(unique_by_ell[ell])} unique diagrams')

# Convenience aliases used downstream
unique_tree = unique_by_ell.get(0, [])
all_unique = [td for ell in range(max_ell + 1) for td in unique_by_ell.get(ell, [])]
print(f'\nTotal unique diagrams across all loop orders: {len(all_unique)}')

---
## 6. Diagram Details & Combinatorial Factor $M(\Gamma)$

For each unique diagram $\Gamma$, display the vertex assignments, edge types,
and the **combinatorial factor** $M(\Gamma)$ together with the scalar prefactor.

The vertex coefficients shown include the $(-1)$ from the partition function
convention $Z = \int \exp(-S)$.

In [ ]:
from msrjd.diagrams.symmetry import classify_coefficient_factors

# Get time-dependence metadata from the model
time_dep_params = HAWKES_MODEL.get('time_dependent_parameters', [])
noise_structure = HAWKES_MODEL.get('noise_structure', {
    'temporal_type': 'white', 'amplitude_params': []
})
print(f'Noise temporal type: {noise_structure.get("temporal_type", "white")}')
print(f'Time-dependent params: {time_dep_params if time_dep_params else "(none -- stationary)"}')
print()


def show_unique_diagram(td, idx, winfo):
    """Display a unique diagram with M(Gamma) and weight structure."""
    M = winfo['M']
    print(f'\n{"="*60}')
    print(f'Unique Diagram {idx}    M = {M}')
    print(f'{"="*60}')

    print('  External legs:')
    for lf, field in sorted(td.external_legs.items()):
        print(f'    leaf {lf} <- {field[0]}{field[1]}')

    if td.vertex_assignments:
        print('  Vertex assignments:')
        for v, vtype in sorted(td.vertex_assignments.items()):
            tname = type(vtype).__name__
            print(f'    v{v} ({tname}): bigrade={vtype.bigrade}, '
                  f'resp={vtype.response_legs}', end='')
            if hasattr(vtype, 'physical_legs'):
                print(f', phys={vtype.physical_legs}', end='')
            print()
            eff_coeff = -SR(vtype.coefficient)
            display(Math(f'\\quad (-1)\\cdot c_{{v_{v}}} = {latex(eff_coeff)}'))

    print('  Edges (propagator assignments):')
    for edge_key in sorted(td.edge_types.keys()):
        resp_leg, phys_leg = td.edge_types[edge_key]
        u, v = edge_key[0], edge_key[1]
        ri, pi = td.propagator_indices.get(edge_key, ('?', '?'))
        print(f'    {u} -> {v}:  {resp_leg[0]}{resp_leg[1]} -> '
              f'{phys_leg[0]}{phys_leg[1]}  [G_hat[{ri},{pi}]]')

    sp = winfo['scalar_prefactor']
    display(Math(
        r'\text{Scalar prefactor: }\;' + latex(sp)
    ))


# ── Display all unique diagrams by loop order ──
for ell in range(max_ell + 1):
    diagrams = unique_by_ell.get(ell, [])
    ell_label = 'TREE-LEVEL' if ell == 0 else f'{ell}-LOOP'
    print('='*60)
    print(f'{ell_label} UNIQUE DIAGRAMS ({len(diagrams)})')
    print('='*60)

    for i, td in enumerate(diagrams, 1):
        winfo = classify_coefficient_factors(td, time_dep_params, noise_structure)
        label = f'Tree-{i}' if ell == 0 else f'{ell}L-{i}'
        show_unique_diagram(td, label, winfo)

---
## 7. Propagator Data & Diagram Prefactors

Assemble the time-domain propagator data (poles, residues, $\delta$
coefficients from cell 1.2) and compute the scalar prefactor
$M(\Gamma) \times \prod_v (-c_v)$ for each unique typed diagram.

No frequency-domain integral construction is used.  The pipeline
goes directly from typed diagrams to time-domain vertex-time
integration.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.  Propagator data + diagram prefactors (pure time-domain path)
# ═══════════════════════════════════════════════════════════════
# This cell assembles everything needed for time-domain evaluation:
#   1. propagator_data: pole_vals, C_mats, D_delta (from cell 8)
#   2. Scalar prefactors for each typed diagram (from symmetry.py)
#   3. The diagram list itself (from cells 15-18)
#
# No frequency-domain integral construction is used.  The frequency-

from msrjd.diagrams.symmetry import classify_coefficient_factors

# ── Propagator data dict ──
propagator_data = {
    'pole_vals': pole_vals,
    'C_mats': C_mats,
    'D_delta': D_delta,
    'nf': nf,

}

print(f'Propagator: {len(pole_vals)} poles, nf={nf}')
print(f'D_delta nonzero entries: {sum(1 for i in range(nf) for j in range(nf) if abs(complex(D_delta[i,j])) > 1e-15)}')

# ── Collect ALL unique typed diagrams across loop orders ──
all_unique = []
for ell in range(max_ell + 1):
    all_unique.extend(unique_by_ell.get(ell, []))
print(f'\nTotal unique diagrams: {len(all_unique)}')

# ── Compute scalar prefactors for each diagram ──
time_dep_params = HAWKES_MODEL.get('time_dependent_parameters', [])
noise_structure = HAWKES_MODEL.get('noise_structure')

diagram_prefactors = []
for i, td in enumerate(all_unique):
    coeff_info = classify_coefficient_factors(
        td,
        time_dep_params=time_dep_params,
        noise_structure=noise_structure,
    )
    diagram_prefactors.append(coeff_info['scalar_prefactor'])

# ── Summary ──
from msrjd.integration.time_domain import _loop_number_from_graph
n_tree = sum(1 for td in all_unique if _loop_number_from_graph(td) == 0)
n_loop = len(all_unique) - n_tree
print(f'  Tree-level: {n_tree}')
print(f'  Loop-level: {n_loop}')

for i, (td, pf) in enumerate(zip(all_unique, diagram_prefactors)):
    ln = _loop_number_from_graph(td)
    D = td.prediagram[0]
    ext = {lf: td.external_legs[lf] for lf in td.prediagram[2]}
    label = f'Tree-{i+1}' if ln == 0 else f'{ln}L-{i+1}'
    print(f'  {label}: pf={pf}, ext={ext}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.4  Numerical evaluation — adaptive by k
# ═══════════════════════════════════════════════════════════════
import numpy as np
from scipy.optimize import fsolve
from sage.all import numerical_integral, CC, fast_callable, CDF, diff

# ═══════════════════════════════════════════════════════════════
# Fundamental parameters (model-agnostic: user specifies these)
# ═══════════════════════════════════════════════════════════════
fundamental = {
    'E':   [1.0, 0.5],                        # external drive per population
    'w':   [[0.3, 0.5], [0.1, 0.3]],         # synaptic weights w_ij
    'tau': 10.0,                               # membrane time constant
}

npop = len(ns.pop)

print('Fundamental parameters:')
for key, val in fundamental.items():
    print(f'  {key} = {val}')

# ═══════════════════════════════════════════════════════════════
# Solve mean-field equations from the model
# ═══════════════════════════════════════════════════════════════
# Read phi_concrete from model and differentiate symbolically
v_sym = SR.var('_v_mf_')
taylor_order = ft.taylor_order

# Build substitution dict: map model parameter symbols to fundamental values.
# This handles any model (linear phi has no 'a', quadratic has 'a', etc.)
_param_subs = {}
for pspec in HAWKES_MODEL.get('parameters', []):
    pname = pspec['name']
    if pname in fundamental:
        if pspec.get('indexed', False):
            for i in ns.pop:
                sym = getattr(ns, pname)[i]
                _param_subs[sym] = fundamental[pname][i]
        else:
            sym = getattr(ns, pname)
            _param_subs[sym] = fundamental[pname]

phi_derivs = {}  # phi_derivs[i][dk] = k-th derivative of phi_i(v), symbolic
for i in ns.pop:
    phi_expr = HAWKES_MODEL['phi_concrete'](ns, i, v_sym)
    phi_derivs[i] = {}
    for dk in range(taylor_order + 1):
        if dk == 0:
            phi_derivs[i][dk] = phi_expr
        else:
            phi_derivs[i][dk] = diff(phi_expr, v_sym, dk)

# Build numerical phi callable from the symbolic expression
def phi_num(i, v_val):
    """Evaluate phi_i(v) numerically."""
    return float(phi_derivs[i][0].subs(_param_subs).subs({v_sym: v_val}))

# Solve MF self-consistency: n*_i = phi_i(E_i + sum_j w_ij * n*_j)
def mf_residual(nstar_vec):
    residuals = []
    for i in ns.pop:
        v_star_i = fundamental['E'][i] + sum(
            fundamental['w'][i][j] * nstar_vec[j] for j in ns.pop
        )
        residuals.append(nstar_vec[i] - phi_num(i, v_star_i))
    return residuals

# Initial guess: small positive rates
nstar_guess = [1.0] * npop
nstar_sol = fsolve(mf_residual, nstar_guess, full_output=True)
nstar_vals = [float(x) for x in nstar_sol[0]]
info = nstar_sol[1]

# Compute v* and phi derivatives at the fixed point
vstar_vals = []
phi_deriv_vals = {}  # phi_deriv_vals[(dk, i)] = d^k phi_i / dv^k |_{v=v*}
for i in ns.pop:
    v_star_i = fundamental['E'][i] + sum(
        fundamental['w'][i][j] * nstar_vals[j] for j in ns.pop
    )
    vstar_vals.append(v_star_i)
    for dk in range(taylor_order + 1):
        phi_deriv_vals[(dk, i)] = float(
            phi_derivs[i][dk].subs(_param_subs).subs({v_sym: v_star_i})
        )

# Sanity check: phi_i(v*_i) should equal n*_i
print(f'\nMean-field solution:')
for i in ns.pop:
    phi_at_vstar = phi_deriv_vals[(0, i)]
    print(f'  pop {i+1}:  v* = {vstar_vals[i]:.6f},  '
          f'n* = {nstar_vals[i]:.6f},  '
          f'phi(v*) = {phi_at_vstar:.6f}  '
          f'{"OK" if abs(phi_at_vstar - nstar_vals[i]) < 1e-10 else "MISMATCH!"}')

print(f'\nDerived phi derivatives:')
for i in ns.pop:
    derivs_str = ', '.join(
        f"phi{dk}_{i+1} = {phi_deriv_vals[(dk, i)]:.6f}"
        for dk in range(1, taylor_order + 1)
        if (dk, i) in phi_deriv_vals
    )
    print(f'  pop {i+1}:  {derivs_str}')

# ═══════════════════════════════════════════════════════════════
# Assemble num_params for symbolic substitution
# ═══════════════════════════════════════════════════════════════
num_params = {}

# Weights
for i in ns.pop:
    for j in ns.pop:
        num_params[SR.var(f'w{i+1}{j+1}')] = fundamental['w'][i][j]

# Timescale
num_params[SR.var('tau')] = fundamental['tau']

# Derived: nstar
for i in ns.pop:
    num_params[ns.nstar[i]] = float(nstar_vals[i])

# Derived: phi derivatives (phi1, phi2, ..., up to taylor_order)
for i in ns.pop:
    for dk in range(1, taylor_order + 1):
        sym = SR.var(f'phi{dk}_{i+1}')
        if (dk, i) in phi_deriv_vals:
            num_params[sym] = phi_deriv_vals[(dk, i)]

print(f'\nFull num_params:')
for sym, val in num_params.items():
    print(f'  {sym} = {val}')

# ── Tau grid for Phase J evaluation ──
tau_out = np.arange(-50.0, 50.05, 0.05)
tau_residue = tau_out.copy()

# Phase I variables set to None (not computed in this notebook)
C_tree_tau = None
C_total_tau = None
C_loop_tau = None
C_tree_residue = None
C_tree_tau_slices = None
C_total_tau_slices = None

print(f'num_params defined ({len(num_params)} entries). Phase I skipped.')


### 8. Time-Domain Integration

For each tree-level diagram, assign a time variable to every vertex,
pin the first external field's time to zero (stationarity), and
integrate over internal vertex times via numerical quadrature.

Each retarded propagator is decomposed as
$G^R(t) = c_\delta \cdot \delta(t) + \Theta(t) \cdot G^{\mathrm{smooth}}(t)$.
The product over edges is expanded into $2^{|E|}$ subsets;
$\delta$ factors are integrated symbolically, and the smooth
remainder is evaluated via `scipy.integrate.quad` / `nquad`.

**Convention**: `total_C(t_1, t_2, ..., t_k)` where position $i$
is the time of `external_fields[i]`.
$\tau_n = t_{n+1} - t_1$, with $t_1 = 0$ (pinned).

Surviving $\delta$ functions (e.g., $\delta(\tau_1)$) are reported
but excluded from the smooth evaluation.  Ito convention: $\Theta(0) = 1$.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.1  Evaluate tree-level contribution via Phase J
# ═══════════════════════════════════════════════════════════════
# Phase J uses explicit numerical quadrature (scipy.integrate.quad /
# nquad) on a fast_callable JIT'd integrand, with polytope bounds
# extracted from the retarded Heaviside factors.  It also enumerates
# the 2^|E| subsets of tree edges whose propagator entries have an
# instantaneous δ(t) component (the \tilde{n} × \delta n coupling in
# the MSR-JD Hawkes action): each subset's δ equalities pin
# integration variables; residual δ on external times comes back as
# a structured shot-noise spike that we insert into the τ grid below.
#
# For k=2 we evaluate on a single 1D τ grid with t_2 pinned to 0.
# For k=3 we evaluate on the two 1D slices that match the simulation
# cell's convention (cell 33):
#   slice 0: vary leaf-1\'s time, fix leaf-0 and leaf-2 at 0
#   slice 1: vary leaf-2\'s time, fix leaf-0 and leaf-1 at 0
# This matches ⟨field_a(0) · field_c(0) · field_b(τ)⟩ (slice 0) and
# ⟨field_a(0) · field_b(0) · field_c(τ)⟩ (slice 1) that the simulation
# extracts from the binned rate time series.
from msrjd.integration.time_domain import (
    compute_correction_td,
    integrate_tree_diagram,
    format_td_integral_latex,
    eval_delta_contributions_on_tau_grid,
)



# ── Utility: compute forbidden τ values where surviving deltas fire ──
# CONVENTION: equality_a from the pipeline is indexed over FREE external
# times (excluding the pinned origin). For k=3 with origin_leaf_idx=0,
        external_fields=external_fields,
# free_ext = [t₂, t₃], so equality_a has length 2 with indices 0, 1.
# All vary_index / fixed_values in these functions use FREE-EXT indexing.

def _forbidden_tau_values(delta_contribs, vary_index_free, fixed_values_free):
    """Return a set of tau values where surviving deltas fire on a 1D slice.
    
    Parameters
    ----------
    delta_contribs : list of dict from _td_result['delta_contributions']
    vary_index_free : int
        Index into the FREE external time vector (0-based). For k=3 with
        origin at leaf 0: 0 = t₂ (τ₁), 1 = t₃ (τ₂).
    fixed_values_free : dict
        {free_ext_index: value} for all non-varying free external times.
    """
    forbidden = set()
    fixed = dict(fixed_values_free) if fixed_values_free else {}
    for dc in delta_contribs:
        eq_a = dc['equality_a']
        eq_c = dc['equality_c']
        n = len(eq_a)
        a_vary = eq_a[vary_index_free] if vary_index_free < n else 0.0
        c_eff = eq_c
        for j in range(n):
            if j != vary_index_free:
                c_eff += eq_a[j] * fixed.get(j, 0.0)
        if abs(a_vary) > 1e-15:
            tau_fire = -c_eff / a_vary
            forbidden.add(round(tau_fire, 10))
        elif abs(c_eff) < 1e-12:
            forbidden.add('ALL')
    return forbidden

def _forbidden_tau_lines_2d(delta_contribs, ax0_free, ax1_free, fixed_values_free):
    """Return list of (a0, a1, c_fixed) lines in 2D where deltas fire.
    
    Parameters use FREE-EXT indexing. Each line: a0*τ_ax0 + a1*τ_ax1 + c = 0.
    """
    lines = []
    fixed = dict(fixed_values_free) if fixed_values_free else {}
    for dc in delta_contribs:
        eq_a = dc['equality_a']
        eq_c = dc['equality_c']
        n = len(eq_a)
        a0 = eq_a[ax0_free] if ax0_free < n else 0.0
        a1 = eq_a[ax1_free] if ax1_free < n else 0.0
        c_f = eq_c
        for j in range(n):
            if j != ax0_free and j != ax1_free:
                c_f += eq_a[j] * fixed.get(j, 0.0)
        if abs(a0) > 1e-15 or abs(a1) > 1e-15:
            lines.append((a0, a1, c_f))
    return lines


if k == 1:
    print(f'Phase J: k=1 mean-field path not exercised — see cell 28 output.')
    tau_phase_j = None
    C_tree_phase_j = None
    C_tree_phase_j_slices = None

elif k == 2:
    _origin_idx = 1  # pin t_2 → 0 so t_1 plays the role of tau
    _td_result = compute_correction_td(
        typed_diagrams=all_unique,
        prefactors=diagram_prefactors,
        propagator_data=propagator_data,
        k=k,
        num_params=num_params,
        origin_leaf_idx=_origin_idx,
    )

    n_tree_groups = sum(
        1 for g in _td_result['groups'] if g['handled_by'] == 'tree_evaluator'
    )
    n_skipped = len(_td_result['skipped_kernel_ids'])
    n_delta_total = len(_td_result['delta_contributions'])
    print(f'Phase J (k={k}): evaluated {n_tree_groups} tree kernel group(s), '
          f'skipped {n_skipped} loop kernel group(s), '
          f'{n_delta_total} shot-noise δ spike(s) produced.')
    for g in _td_result['groups']:
        extra = ''
        if 'n_delta_contributions' in g and g['n_delta_contributions']:
            extra = f"  δ-spikes={g['n_delta_contributions']}"
        print(f"   kernel ell={g['loop_number']}  "
              f"n_diagrams={g['n_diagrams']}  "
              f"handled_by={g['handled_by']}  "
              f"representation={g['representation']}  {g['reason']}{extra}")

    # Display the time-domain integral for each tree kernel group.
    _ext_syms = [SR.var(f't{j+1}_J') for j in range(k)]
    for _gi, _g in enumerate(_td_result['groups']):
        if _g['handled_by'] != 'tree_evaluator':
            continue
        _diag_idx = _g.get('kernel_id')
        if _diag_idx is None or _diag_idx >= len(all_unique):
            continue
        _td_diag = all_unique[_diag_idx]
        _td_pf = diagram_prefactors[_diag_idx]
        _per_group_result = integrate_tree_diagram(
            typed_diagram=_td_diag,
            propagator_data=propagator_data,
            combined_prefactor=_td_pf,
            ext_time_vars=_ext_syms,
            num_params=num_params,
            origin_leaf_idx=_origin_idx,
            external_fields=external_fields,
        )
        _label = f'Diagram {_diag_idx}  (ell={_g["loop_number"]})'
        _latex_str = format_td_integral_latex(
            _per_group_result,
            typed_diagram=_td_diag,
            combined_prefactor=_td_pf,
            ext_time_vars=_ext_syms,
            label=_label,
        )
        display(Math(_latex_str))

    # Evaluate on a single 1D τ grid (t_2 pinned)
    # (Numerical τ-grid evaluation is in cell 8.1c below.)

elif k == 3:
    # k=3: no pinning — Phase J returns C(t_1, t_2, t_3) and we evaluate
    # slices matching the simulation conventions.
    #
    # For each value in TAU_FIXED_LIST we evaluate TWO slices:
    #   slice s=0:  vary leaf 1 time → C(0, τ, τ_fixed)
    #   slice s=1:  vary leaf 2 time → C(0, τ_fixed, τ)
    # τ_fixed = 0 recovers the two canonical "other τ pinned to origin"
    # slices; nonzero τ_fixed lets us peek at off-diagonal cuts of the
    # 2D cumulant surface.
    TAU_FIXED_LIST = [2.0, -2.0]   # can be extended / edited by the user

    _td_result = compute_correction_td(
        typed_diagrams=all_unique,
        prefactors=diagram_prefactors,
        propagator_data=propagator_data,
        k=k,
        num_params=num_params,
        origin_leaf_idx=None,
        external_fields=external_fields,
    )

    n_tree_groups = sum(
        1 for g in _td_result['groups'] if g['handled_by'] == 'tree_evaluator'
    )
    n_skipped = len(_td_result['skipped_kernel_ids'])
    n_delta_total = len(_td_result['delta_contributions'])
    print(f'Phase J (k={k}): evaluated {n_tree_groups} tree kernel group(s), '
          f'skipped {n_skipped} loop kernel group(s), '
          f'{n_delta_total} shot-noise δ spike(s) produced.')
    for g in _td_result['groups']:
        extra = ''
        if 'n_delta_contributions' in g and g['n_delta_contributions']:
            extra = f"  δ-spikes={g['n_delta_contributions']}"
        print(f"   kernel ell={g['loop_number']}  "
              f"n_diagrams={g['n_diagrams']}  "
              f"handled_by={g['handled_by']}  "
              f"representation={g['representation']}  {g['reason']}{extra}")

    # Display the time-domain integral for each tree diagram.
    _ext_syms = [SR.var(f't{j+1}_J') for j in range(k)]
    for _gi, _g in enumerate(_td_result['groups']):
        if _g['handled_by'] != 'tree_evaluator':
            continue
        _diag_idx = _g.get('kernel_id')
        if _diag_idx is None or _diag_idx >= len(all_unique):
            continue
        _td_diag = all_unique[_diag_idx]
        _td_pf = diagram_prefactors[_diag_idx]
        _per_group_result = integrate_tree_diagram(
            typed_diagram=_td_diag,
            propagator_data=propagator_data,
            combined_prefactor=_td_pf,
            ext_time_vars=_ext_syms,
            num_params=num_params,
            origin_leaf_idx=None,
            external_fields=external_fields,
        )
        _label = f'Diagram {_diag_idx}  (ell={_g["loop_number"]})'
        _latex_str = format_td_integral_latex(
            _per_group_result,
            typed_diagram=_td_diag,
            combined_prefactor=_td_pf,
            ext_time_vars=_ext_syms,
            label=_label,
        )
        display(Math(_latex_str))

    # Phase J callable signature is (t_1, t_2, t_3).
    # For each (s, τ_fixed) combo:
    #   s = 0:  vary leaf 1 → C(0, τ, τ_fixed)   (τ_other = t_3 fixed)
    #   s = 1:  vary leaf 2 → C(0, τ_fixed, τ)   (τ_other = t_2 fixed)
    # τ_fixed = 0 gives the canonical slices comparable with the
    # simulation's `τ_other = 0` plots; τ_fixed = +5 probes an
    # off-diagonal cut through the 3-point surface.
    # (Numerical τ-grid evaluation is in cell 8.1c below.)
else:
    print(f'Phase J prototype currently supports k in (2, 3); '
          f'current k={k}.  Nothing to do.')
    tau_phase_j = None
    C_tree_phase_j = None
    C_tree_phase_j_slices = None


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.1b  Symbolic decomposition: δ vs smooth propagator factors
# ═══════════════════════════════════════════════════════════════
# Before numerical integration, show how each tree diagram's integrand
# decomposes when we split each retarded propagator into its δ(t) and
# smooth parts:
#
#   G_R[pi, ri](Δt) = c_δ · δ(Δt) + Θ(Δt) · G_smooth(Δt)
#
# Distributing the product over |E| edges gives 2^|E| terms. For each
# term, δ factors collapse integration variables, and the remaining
# smooth factors define the integrand for numerical quadrature.

from msrjd.integration.time_domain import build_G_t_matrix, G_t_delta_coeff
from msrjd.integration.time_domain.final_integral import _lookup_prop_indices

if k >= 2:
    _t_sym = SR.var('_t_decomp_')
    _G_obj = build_G_t_matrix(propagator_data, _t_sym, num_params=num_params)
    _G_smooth = _G_obj['smooth']
    _G_delta = _G_obj['delta']
    _resp_names = [str(g) for g in ft.ring().gens()][:ft._n_tilde]
    _phys_names = [str(g) for g in ft.ring().gens()][ft._n_tilde:]

    print('Propagator decomposition: G_R[pi, ri](t) = c_δ · δ(t) + Θ(t) · G_smooth(t)')
    print('─' * 70)
    for pi in range(propagator_data['nf']):
        for ri in range(propagator_data['nf']):
            c_d = G_t_delta_coeff(_G_obj, pi, ri)
            smooth_entry = _G_smooth[pi, ri]
            if smooth_entry == 0 and abs(complex(c_d)) < 1e-15:
                continue
            c_d_str = f'{complex(c_d).real:+.4f}' if abs(complex(c_d)) > 1e-15 else '  0   '
            label = f'G[{_phys_names[pi]:>4}, {_resp_names[ri]:>4}]'
            print(f'  {label}:  c_δ = {c_d_str}   smooth(t) = {smooth_entry}')
    print()

    _ext_syms = [SR.var(f't{j+1}') for j in range(k)]
    _origin_idx = 1 if k == 2 else None

    for gi, td in enumerate(all_unique):
        from msrjd.integration.time_domain import _loop_number_from_graph
        if _loop_number_from_graph(td) != 0:
            continue
        D = td.prediagram[0]
        leaves = list(td.prediagram[2])
        leaf_set = set(leaves)
        internal = [v for v in D.vertices() if v not in leaf_set]
        edges = list(D.edges())
        cp_num = float(SR(diagram_prefactors[gi]).subs(num_params))

        vertex_time = {}
        for j, lf in enumerate(leaves):
            t_ext = _ext_syms[j]
            if _origin_idx is not None and j == _origin_idx:
                t_ext = SR(0)
            vertex_time[lf] = t_ext
        int_vars = []
        for v in internal:
            s_v = SR.var(f's_{v}')
            vertex_time[v] = s_v
            int_vars.append(s_v)

        print(f'{"═" * 70}')
        print(f'Kernel group {gi}  (n={g["n_diagrams"]}, pf = {cp_num:+.6f})')
        print(f'{"═" * 70}')
        print(f'  Vertex times: ' + ', '.join(
            f'v{v}={vertex_time[v]}' for v in sorted(vertex_time)))
        print(f'  Integration vars: {[str(s) for s in int_vars]}')
        print(f'  Edges:')

        edge_info = []
        for (u, v, lbl) in edges:
            ri, pi = _lookup_prop_indices(td, (u, v, lbl))
            dt = vertex_time[v] - vertex_time[u]
            c_d = G_t_delta_coeff(_G_obj, pi, ri)
            _t_var = _G_obj.get('t_var', _t_sym)
            smooth_expr = SR(_G_smooth[pi, ri]).subs({_t_var: dt})
            has_delta = abs(complex(c_d)) > 1e-15
            edge_info.append({'u': u, 'v': v, 'ri': ri, 'pi': pi,
                              'dt': dt, 'c_delta': c_d, 'smooth': smooth_expr,
                              'has_delta': has_delta})
            ds = f'c_δ={complex(c_d).real:+.4f}' if has_delta else 'c_δ=0'
            print(f'    ({u}→{v}): G[{_phys_names[pi]},{_resp_names[ri]}]'
                  f'(Δt={dt})  {ds}')

        n_edges = len(edge_info)
        n_with_delta = sum(1 for e in edge_info if e['has_delta'])
        print(f'\n  2^{n_with_delta} = {2**n_with_delta} nonzero subsets '
              f'(from {n_with_delta} edges with δ):')
        print()

        for subset_bits in range(2**n_edges):
            delta_edges = [i for i in range(n_edges) if (subset_bits >> i) & 1]
            smooth_edges = [i for i in range(n_edges) if not ((subset_bits >> i) & 1)]
            if any(not edge_info[i]['has_delta'] for i in delta_edges):
                continue

            delta_coeff = SR(cp_num)
            delta_strs = []
            for i in delta_edges:
                delta_coeff *= SR(complex(edge_info[i]['c_delta']).real)
                delta_strs.append(f'δ({edge_info[i]["dt"]})')

            # Solve δ constraints
            subs = {}
            residuals = []
            remaining = list(int_vars)
            for i in delta_edges:
                dt_sub = SR(edge_info[i]['dt']).subs(subs)
                solved = False
                for iv in remaining:
                    if iv in dt_sub.variables():
                        sol = solve(dt_sub == 0, iv, solution_dict=True)
                        if sol:
                            subs[iv] = sol[0][iv]
                            remaining.remove(iv)
                            solved = True
                            break
                if not solved:
                    residuals.append(str(dt_sub))

            # Classify smooth factors: inner (depends on remaining int vars) vs outer
            remaining_set = set(remaining)
            inner = []
            outer = []
            for i in smooth_edges:
                e = edge_info[i]
                dt_sub = SR(e['dt']).subs(subs)
                deps = set(dt_sub.variables()) & remaining_set
                tag = f'G_sm[{_phys_names[e["pi"]]},{_resp_names[e["ri"]]}]({dt_sub})'
                theta = f'Θ({dt_sub})'
                if deps:
                    inner.append((tag, theta))
                else:
                    outer.append((tag, theta))

            label = 'S=∅' if not delta_edges else f'S={{{",".join(str(i) for i in delta_edges)}}}'
            print(f'  ── {label} ──')
            if delta_strs:
                print(f'     δ:  {" × ".join(delta_strs)}')
            if subs:
                print(f'     →  {", ".join(f"{k_} = {v_}" for k_, v_ in subs.items())}')
            if residuals:
                res_str = ', '.join(residuals)
                if all(r == '0' for r in residuals):
                    print(f'     →  residual: {res_str} = 0 (trivial → degenerate δ on slice)')
                else:
                    print(f'     →  residual ext-time equality: {res_str} = 0')

            coeff_val = float(SR(delta_coeff).real())
            if remaining:
                int_str = ' '.join(f'∫d{s}' for s in remaining)
                if outer:
                    outer_str = ' × '.join(f[0] for f in outer)
                    outer_theta = ' × '.join(f[1] for f in outer)
                    inner_str = ' × '.join(f'{f[0]} × {f[1]}' for f in inner)
                    print(f'     = {coeff_val:+.6f}')
                    print(f'       × [{outer_str}] × [{outer_theta}]')
                    print(f'       × {int_str} [{inner_str}]')
                else:
                    inner_str = ' × '.join(f'{f[0]} × {f[1]}' for f in inner)
                    print(f'     = {coeff_val:+.6f} × {int_str} [{inner_str}]')
            else:
                if outer:
                    outer_str = ' × '.join(f'{f[0]} × {f[1]}' for f in outer)
                    print(f'     = {coeff_val:+.6f} × {outer_str}')
                else:
                    print(f'     = {coeff_val:+.6f}  (constant)')
            print()

    print('─' * 70)
    print('Inspect the above, then run the next cell for numerical evaluation.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.1c  Numerical evaluation of Phase J on the τ grid
# ═══════════════════════════════════════════════════════════════
# This cell evaluates the Phase J callable (built in cell 8.1) at
# each point on the τ grid.  THIS is the slow step — each point
# invokes scipy.integrate.quad on the stripped integrand.
#
# The symbolic decomposition (cell 8.1b) should be inspected first
# to verify the integrand structure looks correct.

if k == 1:
    tau_phase_j = None
    C_tree_phase_j = None
    C_tree_phase_j_slices = None

elif k == 2:
    _total_C_fn = _td_result['total_C']
    tau_phase_j = np.array(tau_residue, dtype=float)
    C_tree_phase_j = np.zeros_like(tau_phase_j, dtype=float)
    for _i, _tv in enumerate(tau_phase_j):
        _val = _total_C_fn(float(_tv), 0.0)
        C_tree_phase_j[_i] = float(np.real(_val))
    # Surviving deltas excluded from smooth evaluation (Ito convention).
    if _td_result['delta_contributions']:
        print(f'  Note: {len(_td_result["delta_contributions"])} surviving '
              f'delta(s) present but excluded from smooth grid.')
    C_tree_phase_j_slices = None
    _mid_j = len(tau_phase_j) // 2
    print(f"Phase J tree  C(0) = {C_tree_phase_j[_mid_j]:.6e}")
    if 'C_tree_residue' in globals() and C_tree_residue is not None and len(C_tree_residue) == len(tau_phase_j):
        _abs_err = float(np.max(np.abs(C_tree_phase_j - C_tree_residue)))
        print(f"max |Phase J - Phase I residue| = {_abs_err:.3e}")

elif k == 3:
    _total_C_fn = _td_result['total_C']
    tau_phase_j = np.array(tau_residue, dtype=float)
    C_tree_phase_j_slices = {}

    # ── Per-group contribution at test points ──
    print()
    print('Per-group smooth contributions:')
    _test_pts = [(0.0, 2.0, 2.0), (0.0, 2.0, -2.0), (0.0, -2.0, 2.0)]
    for _tp in _test_pts:
        print(f'  At {_tp}:')
        _total_at_pt = 0.0
        for _gi, _g_info in enumerate(_td_result['groups']):
            if _g_info.get('contribution') is not None:
                _val = _g_info['contribution'](*[float(x) for x in _tp])
                _total_at_pt += float(complex(_val).real)
                print(f'    Group {_gi}: {float(complex(_val).real):.6e}')
            else:
                print(f'    Group {_gi}: SKIPPED ({_g_info.get("reason","?")})')
        print(f'    TOTAL:    {_total_at_pt:.6e}')
        _total_C_val = _td_result['total_C'](*[float(x) for x in _tp])
        print(f'    total_C:  {float(complex(_total_C_val).real):.6e}')

    for _tau_fixed in TAU_FIXED_LIST:
        for _s in (0, 1):
            _C_slice = np.zeros_like(tau_phase_j, dtype=float)
            for _i, _tv in enumerate(tau_phase_j):
                if _s == 0:
                    _val = _total_C_fn(0.0, float(_tv), float(_tau_fixed))
                else:
                    _val = _total_C_fn(0.0, float(_tau_fixed), float(_tv))
                _C_slice[_i] = float(np.real(_val))
            C_tree_phase_j_slices[(_s, float(_tau_fixed))] = _C_slice

    # Surviving deltas excluded from smooth evaluation (Ito convention).
    if _td_result['delta_contributions']:
        print(f'  Note: {len(_td_result["delta_contributions"])} surviving '
              f'delta(s) present but excluded from smooth slices.')

    C_tree_phase_j = C_tree_phase_j_slices[(0, float(TAU_FIXED_LIST[0]))]
    _mid_j = len(tau_phase_j) // 2
    print()
    for _tau_fixed in TAU_FIXED_LIST:
        print(f"Phase J tree  τ_fixed={_tau_fixed:+g}  "
              f"slice 0 at τ=0 = "
              f"{C_tree_phase_j_slices[(0, float(_tau_fixed))][_mid_j]:.6e}")
        print(f"Phase J tree  τ_fixed={_tau_fixed:+g}  "
              f"slice 1 at τ=0 = "
              f"{C_tree_phase_j_slices[(1, float(_tau_fixed))][_mid_j]:.6e}")

else:
    tau_phase_j = None
    C_tree_phase_j = None
    C_tree_phase_j_slices = None


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.1d  Diagnostic: smooth vs delta on the τ₁=0 slice
# ═══════════════════════════════════════════════════════════════
# For k=3 with external fields [dn₁, dn₁, dn₂], the τ₁=0 slice
# (t₂ = 0, vary t₃ = τ₂) is where the degenerate delta from
# S={edge0, edge1} fires (condition t₁ = t₂ = 0). This cell
# evaluates the smooth and delta contributions SEPARATELY to
# diagnose the theory-sim magnitude gap.
#
# Convention: "slice 0" in cell 8.1c uses
#   total_C(t_a=0, t_b=τ, t_c=τ_fixed)
# So the τ₁=0 slice with τ_fixed=0 is: total_C(0, τ=0, τ₂)
# which corresponds to slice 1 at τ_fixed=0 in cell 8.1c:
#   total_C(0, τ_fixed=0, τ_sweep)
import matplotlib.pyplot as plt

if k == 3 and '_td_result' in dir():
    tau_diag = np.linspace(-50, 50, 501)

    # ── 1. Smooth only: total_C(0, 0, τ₂) ──
    smooth_vals = np.zeros_like(tau_diag)
    for i, t2 in enumerate(tau_diag):
        val = _td_result['total_C'](0.0, 0.0, float(t2))
        smooth_vals[i] = float(np.real(val))

    # ── 2. Delta contributions on this slice ──
    # Slice convention: t₁=0 (pinned origin), t₂=0 (fixed), t₃=τ₂ (sweep)
    # This is: vary_index=2 (leaf 2 = t₃), fixed_values={0: 0, 1: 0}
    from msrjd.integration.time_domain import eval_delta_contributions_on_tau_grid
    delta_vals = eval_delta_contributions_on_tau_grid(
        _td_result['delta_contributions'], tau_diag,
        free_ext_dim=3, vary_index=2,
        fixed_values={0: 0.0, 1: 0.0},
    ).real

    total_vals = smooth_vals + delta_vals

    # ── 3. Print diagnostics ──
    mid = len(tau_diag) // 2
    print(f'τ₁=0 slice diagnostic (t₁=0, t₂=0, vary t₃=τ₂):')
    print(f'  Smooth at τ₂=0:   {smooth_vals[mid]:.6e}')
    print(f'  Delta  at τ₂=0:   {delta_vals[mid]:.6e}')
    print(f'  Total  at τ₂=0:   {total_vals[mid]:.6e}')
    print(f'  Smooth max:        {smooth_vals.max():.6e}')
    print(f'  Smooth min:        {smooth_vals.min():.6e}')
    print(f'  Delta  max:        {delta_vals.max():.6e}')
    print(f'  Delta  min:        {delta_vals.min():.6e}')
    print()

    # Also show the per-group contributions to understand where signal comes from
    print(f'Per-group diagnostics at (0, 0, 5):')
    for gi, g_info in enumerate(_td_result['groups']):
        if g_info.get('contribution') is not None:
            val = g_info['contribution'](0.0, 0.0, 5.0)
            print(f'  Group {gi}: smooth C(0,0,5) = {float(np.real(val)):.6e}')
    print()

    # Per-group at (0, 0, 0)
    print(f'Per-group diagnostics at (0, 0, 0):')
    for gi, g_info in enumerate(_td_result['groups']):
        if g_info.get('contribution') is not None:
            val = g_info['contribution'](0.0, 0.0, 0.0)
            print(f'  Group {gi}: smooth C(0,0,0) = {float(np.real(val)):.6e}')
    print()

    # Show all delta contributions (structured dicts)
    print(f'Delta contributions ({len(_td_result['delta_contributions'])}):')
    for di, dc in enumerate(_td_result['delta_contributions']):
        desc = dc.get('description', '')
        coeff = dc.get('coefficient', '?')
        cond = dc.get('ext_time_equalities', [])
        smooth_fn = dc.get('smooth_factor_fn', None)
        print(f'  [{di}] coeff={coeff}')
        print(f'       ext-time equalities: {cond}')
        print(f'       desc: {desc}')
        if smooth_fn is not None:
            # Evaluate the smooth factor at a few points
            try:
                sf_val = smooth_fn(0.0, 0.0, 5.0)
                print(f'       smooth_factor(0,0,5) = {sf_val}')
            except:
                pass
        print()

    # ── 4. Plot ──
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: Smooth only
    axes[0].plot(tau_diag, smooth_vals, 'b-', lw=1.5)
    axes[0].set_xlabel(r'$\tau_2$', fontsize=13)
    axes[0].set_ylabel(r'^{(3)}$', fontsize=13)
    axes[0].set_title('Smooth only', fontsize=13)
    axes[0].axhline(0, color='k', lw=0.5, ls='--')
    axes[0].axvline(0, color='k', lw=0.5, ls='--')

    # Panel 2: Delta only
    axes[1].plot(tau_diag, delta_vals, 'r-', lw=1.5)
    axes[1].set_xlabel(r'$\tau_2$', fontsize=13)
    axes[1].set_title('Delta contributions only', fontsize=13)
    axes[1].axhline(0, color='k', lw=0.5, ls='--')
    axes[1].axvline(0, color='k', lw=0.5, ls='--')

    # Panel 3: Total (smooth + delta)
    axes[2].plot(tau_diag, total_vals, 'k-', lw=1.5, label='Total (smooth+δ)')
    axes[2].plot(tau_diag, smooth_vals, 'b--', lw=1, alpha=0.7, label='Smooth')
    axes[2].plot(tau_diag, delta_vals, 'r:', lw=1, alpha=0.7, label='Delta')
    axes[2].set_xlabel(r'$\tau_2$', fontsize=13)
    axes[2].set_title('Total = Smooth + Delta', fontsize=13)
    axes[2].legend(fontsize=10)
    axes[2].axhline(0, color='k', lw=0.5, ls='--')
    axes[2].axvline(0, color='k', lw=0.5, ls='--')

    fig.suptitle(r'$\tau_1 = 0$ slice:  $\langle \delta n_1(0) \cdot \delta n_1(0) \cdot \delta n_2(\tau_2) \rangle$',
                 fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Diagnostic only for k=3 with Phase J result.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.2b  Group 0 hand-derived 4-term decomposition vs pipeline
# ═══════════════════════════════════════════════════════════════
# After the canonical remapping, Group 0's contribution(0, τ₁, τ₂)
# always maps:
#   position 0 → external_fields[0] = ('dn',1) at t₁=0 (pinned)
#   position 1 → external_fields[1] = ('dn',1) at t₂=τ₁
#   position 2 → external_fields[2] = ('dn',2) at t₃=τ₂
#
# The hand formula (for the star with source nt₁):
#   C = pf × [ δ(τ₁) × G_sm[dn2,nt1](τ₂)×Θ(τ₂)          (Term 1: S={0,1})
#            + G_sm[dn1,nt1](τ₁)×Θ(τ₁) × G_sm[dn2,nt1](τ₂)×Θ(τ₂)  (Term 2: S={0})
#            + G_sm[dn1,nt1](-τ₁)×Θ(-τ₁) × G_sm[dn2,nt1](τ₂-τ₁)×Θ(τ₂-τ₁) (Term 3: S={1})
#            + ∫ du ... (Term 4: S=∅) ]
#
# Where the propagator entries are determined by the CANONICAL field
# mapping, not by the diagram's internal leaf ordering:
#   Edges to dn1 leaves → G[dn1, nt1]
#   Edge to dn2 leaf → G[dn2, nt1]
import numpy as np
from scipy.integrate import quad
import matplotlib.pyplot as plt

if k == 3 and '_td_result' in dir():
    from msrjd.integration.time_domain import build_G_t_matrix
    from sage.all import fast_callable, CDF, SR

    # ── Build G_smooth entries using CANONICAL field indices ──
    _t_var = SR.var('_t_handcheck_')
    _G_obj = build_G_t_matrix(propagator_data, _t_var, num_params=num_params)
    _G_smooth_mat = _G_obj['smooth']

    # Canonical: external_fields = [('dn',1), ('dn',1), ('dn',2)]
    # All star diagrams with source nt₁ use:
    #   G[dn1, nt1] for the two dn1 edges (resp_idx=0 for nt1)
    #   G[dn2, nt1] for the dn2 edge
    # We need to find the phys_idx for dn1 and dn2 from the field index map
    _pi_dn1 = phys_idx[('dn', 1)]
    _pi_dn2 = phys_idx[('dn', 2)]
    _ri_nt1 = resp_idx[('nt', 1)]

    # Build JIT callables
    _th = SR.var('_th_')
    _Gsm_dn1_nt1 = _G_smooth_mat[_pi_dn1, _ri_nt1].subs({_t_var: _th})
    _Gsm_dn2_nt1 = _G_smooth_mat[_pi_dn2, _ri_nt1].subs({_t_var: _th})
    _Gsm_dn1_fc = fast_callable(_Gsm_dn1_nt1, vars=[_th], domain=CDF)
    _Gsm_dn2_fc = fast_callable(_Gsm_dn2_nt1, vars=[_th], domain=CDF)

    def Gsm_dn1(t):
        return complex(_Gsm_dn1_fc(float(t))).real
    def Gsm_dn2(t):
        return complex(_Gsm_dn2_fc(float(t))).real

    _g0_pf = float(SR(diagram_prefactors[0]).subs(num_params))

    print(f"Group 0 prefactor = {_g0_pf:.6f}")
    print(f"G_smooth[dn1,nt1](0) = {Gsm_dn1(0):.8f}")
    print(f"G_smooth[dn2,nt1](0) = {Gsm_dn2(0):.8f}")

    # ── Evaluate on two slices ──
    _tau = np.linspace(-30, 30, 301)
    _pf = _g0_pf

    # Slice A: τ₁=0, sweep τ₂
    _term1_A = np.array([_pf * Gsm_dn2(t2) if t2 >= 0 else 0.0 for t2 in _tau])

    # Terms 2,3 at τ₁=0: both involve Θ(0) — Ito says Θ(0)=1
    _term2_A = np.array([_pf * Gsm_dn1(0.0) * Gsm_dn2(t2) if t2 >= 0 else 0.0
                         for t2 in _tau])
    _term3_A = np.array([_pf * Gsm_dn1(0.0) * Gsm_dn2(t2) if t2 >= 0 else 0.0
                         for t2 in _tau])

    def _term4_integrand_A(u, tau2):
        if u >= 0 or u >= tau2:
            return 0.0
        return Gsm_dn1(-u)**2 * Gsm_dn2(tau2 - u)

    _term4_A = np.zeros_like(_tau)
    for i, t2 in enumerate(_tau):
        upper = min(0.0, t2)
        if upper <= -200:
            _term4_A[i] = 0.0
        else:
            val, _ = quad(lambda u: _term4_integrand_A(u, t2),
                          -200, upper, limit=200, epsrel=1e-10)
            _term4_A[i] = _pf * val

    # Exclude Term 1 (surviving delta) from the smooth hand sum
    _hand_smooth_A = _term2_A + _term3_A + _term4_A

    # Pipeline Group 0 (canonical ordering)
    _pipeline_g0 = _td_result['groups'][0].get('contribution')
    _pipeline_A = np.zeros_like(_tau)
    if _pipeline_g0 is not None:
        for i, t2 in enumerate(_tau):
            _pipeline_A[i] = float(np.real(_pipeline_g0(0.0, 0.0, float(t2))))

    # Slice B: τ₂=0, sweep τ₁
    _term2_B = np.array([_pf * Gsm_dn1(t1) * Gsm_dn2(0.0) if t1 >= 0 else 0.0
                         for t1 in _tau])
    _term3_B = np.array([_pf * Gsm_dn1(-t1) * Gsm_dn2(-t1) if t1 <= 0 and -t1 >= 0 else 0.0
                         for t1 in _tau])

    def _term4_integrand_B(u, tau1):
        if u >= 0 or u >= tau1:
            return 0.0
        return Gsm_dn1(-u) * Gsm_dn1(tau1 - u) * Gsm_dn2(-u)

    _term4_B = np.zeros_like(_tau)
    for i, t1 in enumerate(_tau):
        upper = min(0.0, t1)
        if upper <= -200:
            _term4_B[i] = 0.0
        else:
            val, _ = quad(lambda u: _term4_integrand_B(u, t1),
                          -200, upper, limit=200, epsrel=1e-10)
            _term4_B[i] = _pf * val

    _hand_smooth_B = _term2_B + _term3_B + _term4_B

    _pipeline_B = np.zeros_like(_tau)
    if _pipeline_g0 is not None:
        for i, t1 in enumerate(_tau):
            _pipeline_B[i] = float(np.real(_pipeline_g0(0.0, float(t1), 0.0)))

    # ── Print diagnostics ──
    _mid = len(_tau) // 2
    print(f"\nSlice A (τ₁=0, sweep τ₂):")
    print(f"  Hand smooth (no delta) at τ₂=5: {_hand_smooth_A[_mid + 25]:.6e}")
    print(f"  Pipeline G0 at τ₂=5:            {_pipeline_A[_mid + 25]:.6e}")
    print(f"\nSlice B (τ₂=0, sweep τ₁):")
    print(f"  Hand smooth at τ₁=5:  {_hand_smooth_B[_mid + 25]:.6e}")
    print(f"  Pipeline G0 at τ₁=5:  {_pipeline_B[_mid + 25]:.6e}")
    print(f"  Hand smooth at τ₁=-5: {_hand_smooth_B[_mid - 25]:.6e}")
    print(f"  Pipeline G0 at τ₁=-5: {_pipeline_B[_mid - 25]:.6e}")

    # ── Plot ──
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0,0].plot(_tau, _hand_smooth_A, 'k-', lw=2, label='Hand smooth (excl δ)')
    axes[0,0].plot(_tau, _pipeline_A, 'r:', lw=2.5, label='Pipeline Group 0')
    axes[0,0].set_title('Slice A (τ₁=0): hand vs pipeline', fontsize=12)
    axes[0,0].set_xlabel('τ₂')
    axes[0,0].legend(fontsize=9)
    axes[0,0].axhline(0, color='k', lw=0.5)

    axes[0,1].plot(_tau, _hand_smooth_A - _pipeline_A, 'r-', lw=1.5)
    axes[0,1].set_title('Slice A: hand − pipeline', fontsize=12)
    axes[0,1].set_xlabel('τ₂')
    axes[0,1].axhline(0, color='k', lw=0.5)

    axes[1,0].plot(_tau, _hand_smooth_B, 'k-', lw=2, label='Hand smooth')
    axes[1,0].plot(_tau, _pipeline_B, 'r:', lw=2.5, label='Pipeline Group 0')
    axes[1,0].set_title('Slice B (τ₂=0): hand vs pipeline', fontsize=12)
    axes[1,0].set_xlabel('τ₁')
    axes[1,0].legend(fontsize=9)
    axes[1,0].axhline(0, color='k', lw=0.5)

    axes[1,1].plot(_tau, _hand_smooth_B - _pipeline_B, 'r-', lw=1.5)
    axes[1,1].set_title('Slice B: hand − pipeline', fontsize=12)
    axes[1,1].set_xlabel('τ₁')
    axes[1,1].axhline(0, color='k', lw=0.5)

    fig.suptitle('Group 0 (canonical ordering): hand smooth vs pipeline',
                 fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Only for k=3 with Phase J result.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.2  Overlay Phase J tree result on the k-point comparison plot
# ═══════════════════════════════════════════════════════════════
# For k=2 this overlays Phase J on top of Phase I\'s FFT IFT and
# residue IFT curves.
# For k=3 we plot Phase J on its own (Phase I was skipped in cell 28)
# as two slice directions × one panel per τ_fixed value. The slice
# dictionary C_tree_phase_j_slices is keyed by (slice_idx, τ_fixed)
# from cell 30.
import matplotlib.pyplot as plt

if k == 2 and C_tree_phase_j is not None:
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))

    if C_tree_tau is not None:
        _mask_fft = np.abs(tau_out) <= 50
        ax.plot(tau_out[_mask_fft], C_tree_tau[_mask_fft].real,
                'b-', lw=2, alpha=0.5, label='Phase I tree (FFT IFT)')

    if 'tau_residue' in globals() and C_tree_residue is not None and len(tau_residue) > 0:
        _mask_r = np.abs(tau_residue) <= 50
        ax.plot(tau_residue[_mask_r], C_tree_residue[_mask_r],
                'g--', lw=2, label='Phase I tree (residue IFT)')

    _mask_j = np.abs(tau_phase_j) <= 50
    ax.plot(tau_phase_j[_mask_j], C_tree_phase_j[_mask_j],
            'm:', lw=2.5, label='Phase J tree (time domain)')

    ax.set_xlabel(r'$\tau = t_1 - t_2$', fontsize=13)
    ax.set_ylabel(r'$C^{(2)}(\tau)$', fontsize=13)
    ax.set_title('Tree-level 2-point correlator — Phase I vs Phase J',
                 fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', lw=0.5)
    plt.tight_layout()
    plt.show()

elif k == 3 and C_tree_phase_j_slices is not None:
    # Group slices by slice index (0 = vary τ₁, 1 = vary τ₂)
    # and plot one subplot per (slice, τ_fixed) pair.
    _tau_fixed_values = sorted({k_[1] for k_ in C_tree_phase_j_slices.keys()})
    _n_fixed = len(_tau_fixed_values)
    fig, axes = plt.subplots(_n_fixed, 2, figsize=(14, 5 * _n_fixed),
                             squeeze=False)
    _mask_j = np.abs(tau_phase_j) <= 50

    _colors = ['m', 'darkorange', 'teal', 'purple']
    for _row, _tau_fixed in enumerate(_tau_fixed_values):
        for _s in (0, 1):
            ax = axes[_row, _s]
            _curve = C_tree_phase_j_slices.get((_s, float(_tau_fixed)))
            if _curve is None:
                continue
            _color = _colors[_row % len(_colors)]
            ax.plot(tau_phase_j[_mask_j], _curve[_mask_j],
                    color=_color, lw=2,
                    label=f'Phase J tree (τ_fixed={_tau_fixed:+g})')
            _vary_idx = _s + 1
            _fixed_idx = 2 - _s
            _xlabel = rf'$\tau_{{{_vary_idx}}}$'
            if _s == 0:
                _title = rf'vary $\tau_1$,  $\tau_2={_tau_fixed:+g}$'
            else:
                _title = rf'vary $\tau_2$,  $\tau_1={_tau_fixed:+g}$'
            ax.set_xlabel(_xlabel, fontsize=13)
            ax.set_ylabel(r'$C^{(3)}$', fontsize=13)
            ax.set_title(_title, fontsize=12)
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3)
            ax.axhline(0, color='k', lw=0.5)
    fig.suptitle(
        'Tree-level 3-point correlator — Phase J time-domain slices',
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()

else:
    print(f'Phase J overlay plot skipped (k={k}).')


### 9. Simulation comparison

Simulate the 2-population nonlinear Hawkes process with the same
fundamental parameters and compare:
- Mean firing rates: simulation vs MF + 1-loop
- Cross-covariance $C_{12}(\tau)$: simulation vs tree + 1-loop

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 9.  Simulation of the 2-population nonlinear Hawkes process
# ═══════════════════════════════════════════════════════════════
import numpy as np
from numpy.random import default_rng

# ── Simulation parameters ──
# For k >= 3 the 3-point cumulant estimator needs MUCH more data than
# the 2-point. The Euler timestep dt_sim can be safely coarsened
# (dt_sim / tau < 0.05 is fine for linear ODE stability/accuracy),
# and the bin width dt_bin can be increased (the signal's timescale
# is ~tau, so dt_bin = 1 still resolves it) for lower per-bin noise.
if k <= 2:
    T_sim    = float(200000)    # 200k — adequate for k=2
    dt_sim   = float(0.02)     # fine Euler step
    dt_bin   = float(0.25)     # fine time resolution
else:
    T_sim    = float(50_000_000)  # 50M — needed for k=3 SNR ~ 3
    dt_sim   = float(0.5)     # coarse but accurate (dt/tau = 5%)
    dt_bin   = float(1.0)     # coarser bins → 4× less shot noise
tau_max  = float(50)       # max lag for covariance
rng      = default_rng(int(42))

# ── Model parameters (from fundamental dict) ──
E_sim   = [float(x) for x in fundamental['E']]
w_sim   = [[float(x) for x in row] for row in fundamental['w']]
tau_sim = float(fundamental['tau'])
# No 'a' parameter for linear phi
npop_sim = len(E_sim)


def phi_sim(v_val):
    """Transfer function: phi(v) = v (linear)."""
    return v_val

import time as _time
# Import the Numba-JIT compiled simulator from a plain .py file.
# (It MUST live in a .py file, not a notebook cell, because SageMath's
# preparser converts integer literals like 0 to Integer(0) which
# Numba's type inference cannot handle.)
from models.hawkes_sim_numba import sim_hawkes_numba


# ═══════════════════════════════════════════════════════════════
# Euler-step simulation with Poisson spike draws (Numba-JIT)
# ═══════════════════════════════════════════════════════════════
n_steps = int(T_sim / dt_sim)
# Fix: use round() so dt_bin / dt_sim near-integers snap correctly,
# then compute the ACTUAL bin width dt_bin_eff = bin_size_steps × dt_sim.
# All downstream analysis uses dt_bin_eff, not the nominal dt_bin.
bin_size_steps = max(int(round(dt_bin / dt_sim)), 1)
dt_bin_eff = bin_size_steps * dt_sim
n_bins = n_steps // bin_size_steps
v_init = np.array([float(vstar_vals[i]) for i in range(npop_sim)])
E_arr = np.array(E_sim, dtype=float)
W_arr = np.array(w_sim, dtype=float)

print(f'Simulating T={T_sim:.0f}, dt_sim={dt_sim}, dt_bin(nominal)={dt_bin}, '
      f'n_steps={n_steps:,}, n_bins={n_bins:,} ...')

# JIT warmup (first call compiles — takes ~2s).
# All args must be plain Python int / float (not Sage Integer).
sim_hawkes_numba(int(1000), float(dt_sim), float(tau_sim), E_arr, W_arr,
                 v_init.copy(), int(bin_size_steps), int(100), int(0))

_t0_sim = _time.perf_counter()
binned_counts, total_spikes = sim_hawkes_numba(
    int(n_steps), float(dt_sim), float(tau_sim), E_arr, W_arr,
    v_init.copy(), int(bin_size_steps), int(n_bins), int(42),
)
_elapsed_sim = _time.perf_counter() - _t0_sim

# T_used excludes any discarded tail from non-divisible n_steps.
T_used = n_bins * dt_bin_eff
sim_rates = total_spikes / T_used
print(f'Done in {_elapsed_sim:.1f}s ({n_steps/_elapsed_sim:,.0f} steps/s).')
if abs(dt_bin_eff - dt_bin) > 1e-12:
    print(f'  NOTE: effective bin width dt_bin_eff = {dt_bin_eff} '
          f'(nominal dt_bin = {dt_bin}, ratio dt_bin/dt_sim = {dt_bin/dt_sim:.4f})')
for i in range(npop_sim):
    print(f'  Pop {i+1}: {int(total_spikes[i])} spikes, rate = {sim_rates[i]:.4f}')


# (The old binned_rates = binned_counts / dt_bin code has been removed.
#  The cumulant estimator now works directly on binned_counts with
#  proper factorial corrections — see models/cumulant_estimator.py.)

# ═══════════════════════════════════════════════════════════════
# Compute k-point cumulant from simulation to match theory
# ═══════════════════════════════════════════════════════════════
max_lag_bins = int(tau_max / dt_bin_eff)
n_fft_sim = 2 * n_bins
tau_sim_grid = np.arange(-max_lag_bins, max_lag_bins + 1) * dt_bin_eff
n_ext = k - 1  # number of independent time differences (stationary)

# Population indices for external fields (0-based)
pop_indices = [ef[1] - 1 for ef in external_fields]
field_labels = [f'{ef[0]}_{ef[1]}' for ef in external_fields]
corr_label = ', '.join(field_labels)

def extract_lag_window(xcorr_full, max_lag_bins, n_fft_total):
    out = np.zeros(2 * max_lag_bins + 1)
    for idx_lag, lag in enumerate(range(-max_lag_bins, max_lag_bins + 1)):
        out[idx_lag] = xcorr_full[lag % n_fft_total]
    return out

if k == 1:
    print(f'k=1: mean rates comparison only')
    C_sim_slices = {}

elif k >= 2:
    # ── General k-point cumulant estimator ──────────────────────────
    # Uses factorial-moment correction for same-population fields at
    # the same time bin: replaces n^m with n(n-1)...(n-m+1), which
    # removes the self-spike shot-noise contribution that would
    # otherwise make the estimator dt_bin-dependent. See
    # models/cumulant_estimator.py for the algorithm.
    from models.cumulant_estimator import compute_kpoint_slice
    from models.cumulant_estimator import compute_kpoint_slice_direct

    # Toggle: True = direct loop (slow but transparent), False = FFT (fast)
    USE_DIRECT = False
    _slice_fn = compute_kpoint_slice_direct if USE_DIRECT else compute_kpoint_slice
    print(f'  Estimator: {"direct (non-FFT)" if USE_DIRECT else "FFT-based"}')

    if 'TAU_FIXED_LIST' not in globals():
        TAU_FIXED_LIST = [0.0] if k == 2 else [0.0, 5.0]

    C_sim_slices = {}
    for tau_fixed in TAU_FIXED_LIST:
        # Snap to the actual bin grid (tau_fixed may not be an exact
        # multiple of dt_bin_eff; use the nearest bin index).
        tau_fixed_bins = int(round(tau_fixed / dt_bin_eff))
        tau_fixed_actual = tau_fixed_bins * dt_bin_eff
        for s in range(n_ext):
            # Build the lag_bins list for this slice.
            # Convention: field a (= external_fields[0]) is the reference
            # (lag = 0). The sweep axis gets lag = None.
            if k == 2:
                lag_bins = [0, None]  # a fixed, b sweeps
            elif k == 3:
                if s == 0:
                    # vary tau1 (= lag of field b from a), fix tau2 = tau_fixed
                    lag_bins = [0, None, tau_fixed_bins]
                else:
                    # vary tau2 (= lag of field c from a), fix tau1 = tau_fixed
                    lag_bins = [0, tau_fixed_bins, None]
            else:
                raise NotImplementedError(f'k={k} slice layout not yet configured')

            tau_sim_grid, C_slice = _slice_fn(
                binned_counts, float(dt_bin_eff),
                [int(p) for p in pop_indices],
                lag_bins, int(max_lag_bins),
            )
            if k == 2:
                C_sim_slices[s] = C_slice
            else:
                # Mask sim data at tau values where surviving deltas fire
                # (the factorial correction doesn't remove the k>=3 shot noise)
                # Use FREE-EXT indexing: 0 = t₂ (τ₁), 1 = t₃ (τ₂)
                if '_td_result' in dir() and _td_result.get('delta_contributions'):
                    _vary_free = 0 if s == 0 else 1  # free-ext index
                    _fixed_free = 1 if s == 0 else 0
                    _fixed_d_sim = {_fixed_free: float(tau_fixed)}
                    _forbidden_sim = _forbidden_tau_values(
                        _td_result['delta_contributions'], _vary_free, _fixed_d_sim)
                    _dtau_sim = abs(tau_sim_grid[1] - tau_sim_grid[0]) if len(tau_sim_grid) > 1 else 1.0
                    for _ftv in _forbidden_sim:
                        if _ftv == 'ALL':
                            C_slice[:] = np.nan
                        else:
                            # Mask ±2 bins around the delta to catch the full spike
                            _mask_radius = max(2.0 * _dtau_sim, float(dt_bin_eff))
                            _near_mask = np.abs(tau_sim_grid - float(_ftv)) <= _mask_radius
                            C_slice[_near_mask] = np.nan
                C_sim_slices[(s, float(tau_fixed))] = C_slice

    print(f'Cumulant slices computed (k={k}, '
          f'{len(C_sim_slices)} slice(s), factorial-corrected).')


print(f'\nSimulated rates: {[f"{r:.4f}" for r in sim_rates]}')
print(f'MF rates:        {[f"{nstar_vals[i]:.4f}" for i in range(npop_sim)]}')


# ═══════════════════════════════════════════════════════════════
# Comparison plots
# ═══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

# For k=1, k=2: single slice comparison + rates panel.
# For k=3: one row per τ_fixed value (so the user can see how both
# slice directions compare against Phase J at several fixed values).

mf_rates = [float(nstar_vals[i]) for i in range(npop_sim)]

def _plot_mean_rates(ax):
    x_pos = np.arange(npop_sim)
    width = 0.25
    ax.bar(x_pos - width, sim_rates, width, label='Simulation',
           color='#4CAF50', edgecolor='k', linewidth=0.5)
    ax.bar(x_pos, mf_rates, width, label='Mean-field',
           color='#2196F3', edgecolor='k', linewidth=0.5)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'Pop {i+1}' for i in range(npop_sim)])
    ax.set_ylabel('Firing rate', fontsize=12)
    ax.set_title('Mean firing rates', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')


if n_ext <= 1:
    # k=1 or k=2: single row with rates + one cumulant panel
    n_plots = max(1, n_ext)
    fig, axes = plt.subplots(1, 1 + n_plots,
                             figsize=(7 * (1 + n_plots), 5))
    if not isinstance(axes, np.ndarray):
        axes = [axes]
    _plot_mean_rates(axes[0])

    for s in range(n_ext):
        ax = axes[1 + s]
        # Old schema (k=2): C_sim_slices keyed by s alone
        _sim = C_sim_slices.get(s)
        if _sim is not None:
            ax.plot(tau_sim_grid, _sim, 'k-', lw=1.5, alpha=0.7,
                    label='Simulation')

        mask_an = np.abs(tau_out) <= tau_max
        if C_tree_tau is not None:
            ax.plot(tau_out[mask_an], C_tree_tau[mask_an].real,
                    'b-', lw=2, label='Phase I tree')
        if C_loop_tau is not None:
            ax.plot(tau_out[mask_an], C_total_tau[mask_an].real,
                    'r--', lw=2, label='Phase I tree + 1-loop')

        if tau_phase_j is not None and C_tree_phase_j is not None:
            _mask_j = np.abs(tau_phase_j) <= tau_max
            ax.plot(tau_phase_j[_mask_j], C_tree_phase_j[_mask_j],
                    'm:', lw=2.5, label='Phase J tree (time domain)')

        ax.set_xlabel(r'$\tau = t_1 - t_2$', fontsize=13)
        is_auto = (len(external_fields) >= 2
                   and external_fields[0] == external_fields[1])
        ctype = 'Auto' if is_auto else 'Cross'
        ax.set_title(f'{ctype}-covariance: {corr_label}', fontsize=13)
        ax.set_ylabel(rf'$C^{{({k})}}$', fontsize=13)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.axhline(0, color='k', lw=0.5)

    plt.tight_layout()
    plt.show()

else:
    # k>=3: one row per τ_fixed value. Row 0 also carries the mean
    # rates subplot in the leftmost column; later rows leave that slot
    # blank.
    _tau_fixed_values = sorted({k_[1] for k_ in C_sim_slices.keys()}) \
        if C_sim_slices else [0.0]
    _n_rows = len(_tau_fixed_values)
    fig, axes = plt.subplots(_n_rows, 1 + n_ext,
                             figsize=(7 * (1 + n_ext), 5 * _n_rows),
                             squeeze=False)
    _plot_mean_rates(axes[0, 0])
    # Leave any remaining leftmost slots blank
    for _r in range(1, _n_rows):
        axes[_r, 0].axis('off')

    for _r, _tau_fixed in enumerate(_tau_fixed_values):
        for s in range(n_ext):
            ax = axes[_r, 1 + s]

            # Simulation
            _sim = C_sim_slices.get((s, float(_tau_fixed)))
            if _sim is not None:
                ax.plot(tau_sim_grid, _sim, 'k-', lw=1.5, alpha=0.7,
                        label='Simulation')

            # Phase J theory (time-domain slices)
            _pj_key = (s, float(_tau_fixed))
            if ('C_tree_phase_j_slices' in globals()
                    and C_tree_phase_j_slices is not None
                    and _pj_key in C_tree_phase_j_slices
                    and tau_phase_j is not None):
                _mask_j = np.abs(tau_phase_j) <= tau_max
                ax.plot(tau_phase_j[_mask_j],
                        C_tree_phase_j_slices[_pj_key][_mask_j],
                        'm-', lw=2, label='Phase J tree (time domain)')

            # Phase I is skipped for k>=3 (see cell 28).

            other_label = (rf'$\tau_{{{2-s}}}={_tau_fixed:+g}$')
            ax.set_xlabel(rf'$\tau_{{{s+1}}}$', fontsize=13)
            ax.set_title(f'$k={k}$ slice: vary $\\tau_{{{s+1}}}$, '
                         f'{other_label}', fontsize=12)
            ax.set_ylabel(rf'$C^{{({k})}}$', fontsize=13)
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3)
            ax.axhline(0, color='k', lw=0.5)

    plt.tight_layout()
    plt.show()

# ── Summary ──
print(f'\n{"="*60}')
print('Comparison summary')
print(f'{"="*60}')
print(f'  k = {k}, external_fields = {external_fields}')
for i in range(npop_sim):
    mf = nstar_vals[i]
    sim_r = sim_rates[i]
    pct = 100 * (sim_r - mf) / mf if mf > 0 else 0
    print(f'  Pop {i+1}:  sim={sim_r:.4f}  MF={mf:.4f}  diff={pct:+.2f}%')
print(f'\n  T={T_sim:.0f}, dt={dt_sim}, bins={n_bins}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.3  Heatmap comparison: theory vs simulation on the (τ₁, τ₂) grid
# ═══════════════════════════════════════════════════════════════
# Both theory and sim heatmaps are built row-by-row using the SAME
# convention: fix τ₁ (= lag of field b from field a), sweep τ₂
# (= lag of field c from field a). This ensures identical lag
# conventions and normalization between theory and sim.

import matplotlib.pyplot as plt

if k == 3 and '_td_result' in dir():
    # ── Settings ──
    HEATMAP_TAU_MAX  = 30.0
    HEATMAP_TAU_STEP = 1.0

    tau_1d = np.arange(-HEATMAP_TAU_MAX, HEATMAP_TAU_MAX + HEATMAP_TAU_STEP/2,
                       HEATMAP_TAU_STEP)
    n_tau = len(tau_1d)

    # ── Theory heatmap: row-by-row, same convention as sim ──
    # For each τ₁ row, evaluate total_C(t_a=0, t_b=τ₁, t_c=τ₂) for
    # each τ₂ in tau_1d.
    _total_C_fn = _td_result['total_C']
    C_theory_2d = np.zeros((n_tau, n_tau))
    print(f'Evaluating theory heatmap ({n_tau}x{n_tau} = {n_tau**2} points)...')
    for i, t1 in enumerate(tau_1d):
        for j, t2 in enumerate(tau_1d):
            C_theory_2d[i, j] = float(np.real(
                _total_C_fn(0.0, float(t1), float(t2))
            ))
    print(f'  Smooth only. Range: [{C_theory_2d.min():.4e}, {C_theory_2d.max():.4e}]')

    # Surviving deltas excluded from smooth heatmap (Ito convention).
    if _td_result.get('delta_contributions'):
        print(f'  Note: {len(_td_result["delta_contributions"])} surviving '
              f'delta(s) present but excluded from smooth heatmap.')

    # ── Sim heatmap: row-by-row using compute_kpoint_slice ──
    # For each τ₁ row, call compute_kpoint_slice with:
    #   lag_bins = [0, t1_bins, None]   (field a at 0, b at τ₁, c sweeps)
    from models.cumulant_estimator import compute_kpoint_slice
    heatmap_lag_bins = int(HEATMAP_TAU_MAX / dt_bin_eff)

    C_sim_2d = np.zeros((n_tau, n_tau))
    print(f'Evaluating sim heatmap ({n_tau} rows)...')
    for i, t1_val in enumerate(tau_1d):
        t1_bins = int(round(t1_val / dt_bin_eff))
        lag_bins_row = [0, t1_bins, None]
        try:
            tau_row, c_row = compute_kpoint_slice(
                binned_counts, float(dt_bin_eff),
                [int(p) for p in pop_indices],
                lag_bins_row, heatmap_lag_bins,
            )
            C_sim_2d[i, :] = np.interp(tau_1d, tau_row, c_row)
        except Exception as e:
            print(f'  row {i} (tau1={t1_val}): {e}')
            C_sim_2d[i, :] = 0.0
    print(f'  Done. Range: [{np.nanmin(C_sim_2d):.4e}, {np.nanmax(C_sim_2d):.4e}]')

    # Mask sim heatmap along delta-firing lines (free-ext indices 0, 1)
    if '_td_result' in dir() and _td_result.get('delta_contributions'):
        _delta_lines = _forbidden_tau_lines_2d(
            _td_result['delta_contributions'], 0, 1, {})
        for (a1, a2, cf) in _delta_lines:
            for i, t1 in enumerate(tau_1d):
                for j, t2 in enumerate(tau_1d):
                    if abs(a1 * t1 + a2 * t2 + cf) < HEATMAP_TAU_STEP * 1.0:
                        C_sim_2d[i, j] = np.nan
        print(f'  Masked {len(_delta_lines)} delta line(s) in sim heatmap.')

    # ── 3-panel plot ──
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    extent = [tau_1d[0], tau_1d[-1], tau_1d[0], tau_1d[-1]]

    # Panel 1: Theory on its own scale
    th_max = max(abs(np.nanmin(C_theory_2d)), abs(np.nanmax(C_theory_2d)), 1e-15)
    im0 = axes[0].imshow(
        C_theory_2d.T, origin='lower', aspect='equal', extent=extent,
        cmap='RdBu_r', vmin=-th_max, vmax=th_max,
    )
    axes[0].set_xlabel(r'$\tau_1$', fontsize=13)
    axes[0].set_ylabel(r'$\tau_2$', fontsize=13)
    axes[0].set_title('Theory (tree-level)', fontsize=13)
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    # Panel 2: Sim on its own scale (shows the shot-noise ridge)
    sim_max = max(abs(np.nanmin(C_sim_2d)), abs(np.nanmax(C_sim_2d)), 1e-15)
    im1 = axes[1].imshow(
        C_sim_2d.T, origin='lower', aspect='equal', extent=extent,
        cmap='RdBu_r', vmin=-sim_max, vmax=sim_max,
    )
    axes[1].set_xlabel(r'$\tau_1$', fontsize=13)
    axes[1].set_title('Sim (full scale)', fontsize=13)
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    # Panel 3: Sim clamped to THEORY scale (reveals smooth structure)
    im2 = axes[2].imshow(
        C_sim_2d.T, origin='lower', aspect='equal', extent=extent,
        cmap='RdBu_r', vmin=-th_max, vmax=th_max,
    )
    axes[2].set_xlabel(r'$\tau_1$', fontsize=13)
    axes[2].set_title('Sim (theory scale)', fontsize=13)
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    fig.suptitle(rf'$k=3$ cumulant heatmap:  $\langle {corr_label} \rangle$',
                 fontsize=14)
    plt.tight_layout()
    plt.show()

else:
    print(f'Heatmap comparison only for k=3 (current k={k}).')
